# Pipeline de experimentos PLN

Usando subconjunto dos nossos dados (rotulação ainda provisória).

Estou montando meu Google Drive, pois estou lendo o arquivo de entrada de lá.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## 1. Estruturação dos dados

Primeiro, vamos importar o conjunto de dados.

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

import pandas as pd
dados = pd.read_csv('./drive/My Drive/Colab Notebooks/Posicionamento/aborto-consolidated-parent-based.tsv', sep='\t', decimal = ',', encoding = 'UTF-8')
dados

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,target_id,target_message,id,parent_id,author,parent_name,parent_message,message,parent_label,target_parent_message,target_and_message,label
0,1,A paternidade deveria ser considerada nas disc...,Alvo,Alvo,Alvo,Alvo,Alvo,A paternidade deveria ser considerada nas disc...,Alvo da conversa,A paternidade deveria ser considerada nas disc...,A paternidade deveria ser considerada nas disc...,Alvo da conversa
1,1,A paternidade deveria ser considerada nas disc...,1,NaN,Barbara_Charles,Raiz,NaN,"E a paternidade, Sra. Ministra ? Considerando ...",NaN,A paternidade deveria ser considerada nas disc...,A paternidade deveria ser considerada nas disc...,NaN
2,1,A paternidade deveria ser considerada nas disc...,2,1,Mary_Lane,Barbara_Charles,"E a paternidade, Sra. Ministra ? Considerando ...","É engraçado que, se formos seguir a lógica do ...",NaN,A paternidade deveria ser considerada nas disc...,A paternidade deveria ser considerada nas disc...,Concorda
3,1,A paternidade deveria ser considerada nas disc...,3,2,Karen_Poole,Mary_Lane,"É engraçado que, se formos seguir a lógica do ...","Na hora de abortar é escolha da mulher, na hor...",Concorda,A paternidade deveria ser considerada nas disc...,A paternidade deveria ser considerada nas disc...,Concorda
4,1,A paternidade deveria ser considerada nas disc...,4,3,Barbara_Charles,Karen_Poole,"Na hora de abortar é escolha da mulher, na hor...",Sabe qual é o bizarro em um contexto onde o ab...,Concorda,A paternidade deveria ser considerada nas disc...,A paternidade deveria ser considerada nas disc...,Concorda
...,...,...,...,...,...,...,...,...,...,...,...,...
776,75,Quando descobrimos bactérias em Marte todo mun...,11,1,Shawn_Bell,Kimberly_Elliott,Quando descobrem uma bactéria aleatória em Mar...,É pq um dos 2 ja se desenvolveu e nasceu,Comentário Original,Quando descobrimos bactérias em Marte todo mun...,Quando descobrimos bactérias em Marte todo mun...,Discorda
777,75,Quando descobrimos bactérias em Marte todo mun...,12,1,Jason_Moore,Kimberly_Elliott,Quando descobrem uma bactéria aleatória em Mar...,"A questão não é se algo constitui vida ou não,...",Comentário Original,Quando descobrimos bactérias em Marte todo mun...,Quando descobrimos bactérias em Marte todo mun...,Discorda
778,75,Quando descobrimos bactérias em Marte todo mun...,13,12,Kimberly_Elliott,Jason_Moore,"A questão não é se algo constitui vida ou não,...",não basta possuir vida para ter direitos human...,Discorda,Quando descobrimos bactérias em Marte todo mun...,Quando descobrimos bactérias em Marte todo mun...,Concorda
779,75,Quando descobrimos bactérias em Marte todo mun...,14,13,Jason_Moore,Kimberly_Elliott,não basta possuir vida para ter direitos human...,Você é mais burro do que eu pensava. Queria te...,Concorda,Quando descobrimos bactérias em Marte todo mun...,Quando descobrimos bactérias em Marte todo mun...,Discorda


Agora, vamos definir as variáveis de interesse (ou seja, a variável texto e a categoria que será prevista). Além disso, vamos utilizar o *Counter* para avaliar se há desbalanceamento entre as classes.

In [ ]:
from math import nan
from collections import Counter

dados = dados[(dados['label'] != 'Alvo da conversa') & (dados['label'] != 'Comentário Original') & (dados['label'].notna())]


dados['label'] = dados['label'].replace('Discute', 'Outros')
dados['label'] = dados['label'].replace('Irrelevante', 'Outros')
dados['label'] = dados['label'].replace('Pede Informações', 'Outros')

dados['parent_label'] = dados['parent_label'].replace('Discute', 'Outros')
dados['parent_label'] = dados['parent_label'].replace('Irrelevante', 'Outros')
dados['parent_label'] = dados['parent_label'].replace('Pede Informações', 'Outros')

X = dados['target_and_message']
y = dados['label']
#y = dados['target_id']

Counter(y)

<ipython-input-3-82746e4f3de5>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dados['label'] = dados['label'].replace('Discute', 'Outros')
<ipython-input-3-82746e4f3de5>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dados['label'] = dados['label'].replace('Irrelevante', 'Outros')
<ipython-input-3-82746e4f3de5>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https

Counter({'Concorda': 228, 'Outros': 168, 'Discorda': 235})

Por fim, vamos dividir o conjunto de dados em duas partições: a de treinamento e a de teste. Isso será feito utilizando o comando *stratify* para que sejam mantidas as mesmas proporções entre as classes no conjunto de treinamento e de teste.

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.202, shuffle=False)

Counter(y_test)

Counter({'Outros': 18, 'Concorda': 47, 'Discorda': 63})

## 2. Criação de modelos

Agora, vamos criar os modelos que serão testados na última etapa. Primeiro, vamos importar as bibliotecas que utilizaremos:

In [ ]:
# Pipelines
from sklearn.pipeline import Pipeline

# Engenharia de Características (features)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler

# Modelos
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import MultinomialNB

from sklearn import svm

# Métricas
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix

Não foi utilizado Grid Search ainda (para otimização de hiperparâmetros).

In [ ]:
from unicodedata import normalize

minhaStopList = [normalize('NFKD', n).encode('ASCII', 'ignore').decode('ASCII') for n in stopwords.words('portuguese')]
#minhaStopList.append('ii')


In [ ]:
baseline = DummyClassifier(strategy='most_frequent', random_state = 100, constant = None)
reglog = LogisticRegression(class_weight='balanced', max_iter=2000, solver='liblinear', penalty='l2', C=1.3225)
mlp = MLPClassifier(activation='relu', max_iter=2000)
NB = MultinomialNB()

SVM = svm.SVC(kernel='poly', C = 1, gamma=1)

## 3. Comparação dos modelos

Agora, vamos criar um pipeline de pipelines para comparar diferentes combinações de modelos e pré-processamentos diferentes. Abaixo, crio uma lista com todas as combinações que quero testar.

OBS.: É importante dar um nome bom para cada uma das etapas; isso vai ajudar a identificar o modelo em questão na fase de comparações.

In [ ]:
# modelos1: conjunto de modelos que não usam seleção de atributos
modelos1 = [Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('dummy', baseline)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('NB', NB)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('NB', NB)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('relog', reglog)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('SVM', SVM)]),
           ]

# modelos2: conjunto de modelos que usam seleção de atributos com a função SelectKBest
atributos  = 300
modelos2 = [Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('dummy', baseline)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('reglog', reglog)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('reglog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('KBest', SelectKBest(chi2, k=atributos)),
                              ('SVM', SVM)]),
           ]

# modelos3: conjunto de modelos que utilizam redução de dimensionalidade usando Análise de Componentes Principais (PCA)
modelos3 = [Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('pca', TruncatedSVD(atributos)),
                              ('dummy', baseline)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bag-of-words', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList)),
                              ('pca', TruncatedSVD(atributos)),
                              ('reglog', reglog)]),
           Pipeline(steps=[('tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,)),
                              ('pca', TruncatedSVD(atributos)),
                              ('reglog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)), ('Normalizing',MinMaxScaler()),
                              ('NB', NB)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('relog', reglog)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('mlp', mlp)]),
           Pipeline(steps=[('bigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram', CountVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('bigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(2,2))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(3,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           Pipeline(steps=[('uni-bi-trigram tf-idf', TfidfVectorizer(strip_accents='ascii', lowercase=True, stop_words=minhaStopList,ngram_range=(1,3))),
                              ('pca', TruncatedSVD(atributos)),
                              ('SVM', SVM)]),
           ]

# Cada bloco a seguir avalia um conjunto de modelos
Os modelos são avaliados considerando validação cruzada utilizando o conjunto de treinamento.

# Modelos que não utilizam nenhuma abordagem de seleção de atributos/redução de dimensionalidade.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=3, random_state = 123, shuffle = True)

testados = modelos1

for m in testados:
  print([i for i,j in m.steps], '\tf1_macro\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='f1_macro').mean(), 3))
  print([i for i,j in m.steps], '\taccuracy\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='accuracy').mean(), 3))
  #print([i for i,j in m.steps], '\troc_auc\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='roc_auc').mean(), 3))

['bag-of-words', 'dummy'] 	f1_macro	 0.176
['bag-of-words', 'dummy'] 	accuracy	 0.36
['bag-of-words', 'mlp'] 	f1_macro	 0.395
['bag-of-words', 'mlp'] 	accuracy	 0.398
['tf-idf', 'mlp'] 	f1_macro	 0.401
['tf-idf', 'mlp'] 	accuracy	 0.4
['bag-of-words', 'NB'] 	f1_macro	 0.474
['bag-of-words', 'NB'] 	accuracy	 0.469
['tf-idf', 'NB'] 	f1_macro	 0.46
['tf-idf', 'NB'] 	accuracy	 0.459
['bag-of-words', 'SVM'] 	f1_macro	 0.395
['bag-of-words', 'SVM'] 	accuracy	 0.402
['tf-idf', 'SVM'] 	f1_macro	 0.41
['tf-idf', 'SVM'] 	accuracy	 0.411
['bag-of-words', 'relog'] 	f1_macro	 0.418
['bag-of-words', 'relog'] 	accuracy	 0.417
['tf-idf', 'relog'] 	f1_macro	 0.473
['tf-idf', 'relog'] 	accuracy	 0.469
['bigram', 'NB'] 	f1_macro	 0.433
['bigram', 'NB'] 	accuracy	 0.433
['trigram', 'NB'] 	f1_macro	 0.423
['trigram', 'NB'] 	accuracy	 0.423
['uni-bi-trigram', 'NB'] 	f1_macro	 0.46
['uni-bi-trigram', 'NB'] 	accuracy	 0.457
['bigram tf-idf', 'NB'] 	f1_macro	 0.433
['bigram tf-idf', 'NB'] 	accuracy	 0.431
['tr

# Modelos que utilizam seleção de atributos (utilizando a função SelectKBest).

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=3, random_state = 123, shuffle = True)

testados = modelos2

for m in testados:
  print([i for i,j in m.steps], '\tf1_macro\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='f1_macro').mean(), 3))
  print([i for i,j in m.steps], '\taccuracy\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='accuracy').mean(), 3))
  #print([i for i,j in m.steps], '\troc_auc\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='roc_auc').mean(), 3))

['bag-of-words', 'KBest', 'dummy'] 	f1_macro	 0.176
['bag-of-words', 'KBest', 'dummy'] 	accuracy	 0.36
['bag-of-words', 'KBest', 'mlp'] 	f1_macro	 0.398
['bag-of-words', 'KBest', 'mlp'] 	accuracy	 0.382
['tf-idf', 'KBest', 'mlp'] 	f1_macro	 0.402
['tf-idf', 'KBest', 'mlp'] 	accuracy	 0.394
['bag-of-words', 'KBest', 'NB'] 	f1_macro	 0.44
['bag-of-words', 'KBest', 'NB'] 	accuracy	 0.439
['tf-idf', 'KBest', 'NB'] 	f1_macro	 0.404
['tf-idf', 'KBest', 'NB'] 	accuracy	 0.411
['bag-of-words', 'KBest', 'SVM'] 	f1_macro	 0.414
['bag-of-words', 'KBest', 'SVM'] 	accuracy	 0.417
['tf-idf', 'KBest', 'SVM'] 	f1_macro	 0.202
['tf-idf', 'KBest', 'SVM'] 	accuracy	 0.368
['bag-of-words', 'KBest', 'reglog'] 	f1_macro	 0.447
['bag-of-words', 'KBest', 'reglog'] 	accuracy	 0.447
['tf-idf', 'KBest', 'reglog'] 	f1_macro	 0.428
['tf-idf', 'KBest', 'reglog'] 	accuracy	 0.425
['bigram', 'KBest', 'NB'] 	f1_macro	 0.374
['bigram', 'KBest', 'NB'] 	accuracy	 0.376
['trigram', 'KBest', 'NB'] 	f1_macro	 0.34
['trigram

# Modelos que utilizam redução de dimensionalidade usando Análise de Componentes Principais (PCA)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=3, random_state = 123, shuffle = True)

testados = modelos3

for m in testados:
  print([i for i,j in m.steps], '\tf1_macro\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='f1_macro').mean(), 3))
  print([i for i,j in m.steps], '\taccuracy\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='accuracy').mean(), 3))
  #print([i for i,j in m.steps], '\troc_auc\t', round(cross_val_score(m, x_train, y_train, cv=kf, scoring='roc_auc').mean(), 3))

['bag-of-words', 'pca', 'dummy'] 	f1_macro	 0.176
['bag-of-words', 'pca', 'dummy'] 	accuracy	 0.36
['bag-of-words', 'pca', 'mlp'] 	f1_macro	 0.397
['bag-of-words', 'pca', 'mlp'] 	accuracy	 0.396
['tf-idf', 'pca', 'mlp'] 	f1_macro	 0.399
['tf-idf', 'pca', 'mlp'] 	accuracy	 0.406
['bag-of-words', 'pca', 'Normalizing', 'NB'] 	f1_macro	 0.24
['bag-of-words', 'pca', 'Normalizing', 'NB'] 	accuracy	 0.37
['tf-idf', 'pca', 'Normalizing', 'NB'] 	f1_macro	 0.271
['tf-idf', 'pca', 'Normalizing', 'NB'] 	accuracy	 0.374
['bag-of-words', 'pca', 'SVM'] 	f1_macro	 0.411
['bag-of-words', 'pca', 'SVM'] 	accuracy	 0.41
['tf-idf', 'pca', 'SVM'] 	f1_macro	 0.4
['tf-idf', 'pca', 'SVM'] 	accuracy	 0.402
['bag-of-words', 'pca', 'reglog'] 	f1_macro	 0.429
['bag-of-words', 'pca', 'reglog'] 	accuracy	 0.417
['tf-idf', 'pca', 'reglog'] 	f1_macro	 0.473
['tf-idf', 'pca', 'reglog'] 	accuracy	 0.467
['bigram', 'pca', 'Normalizing', 'NB'] 	f1_macro	 0.25
['bigram', 'pca', 'Normalizing', 'NB'] 	accuracy	 0.358
['trigr

Os diferentes modelos foram avaliados com validação cruzada (usando apenas o conjunto de treinamento). O melhor "deve" ser o selecionado e seu desempenho deverá ser avaliado com o conjunto de teste.

## 4. Teste do modelo escolhido

Testa-se o modelo escolhido (teoricamente o que obteve o melhor desempenho). O modelo é treinado com todo o conjunto de treinamento e testado com o conjunto de teste.

In [ ]:
from sklearn import metrics # Métricas para avaliação da classificação

modelos1[3].fit(x_train, y_train)

preds = modelos1[3].predict(x_test) ### escolha um modelo para verificar seu desempenho no conjunto de teste
report=metrics.classification_report(y_test,preds)
print(report)


              precision    recall  f1-score   support

    Concorda       0.41      0.66      0.51        47
    Discorda       0.50      0.32      0.39        63
      Outros       0.15      0.11      0.13        18

    accuracy                           0.41       128
   macro avg       0.36      0.36      0.34       128
weighted avg       0.42      0.41      0.40       128



Pipeline de experimentos adaptado de notebook desenvolvido pela ex-aluna Laís Carraro.